# 10年定着予測 - KNN局所ターゲットエンコーディング + Cox生存時間特徴量（60_）

## 位置づけ

`54_`（R0_memofix_plus_LM、444列、Public 0.515030）をベースラインに、2つの新しい軸を
**独立した特徴量**として追加検証する。どちらも「弱いモデルを最終予測器/ブレンド相手にせず、
特徴量生成器として使う」という同じ設計原則に基づく（`31_`/`32_`で線形モデル単体・確率ブレンドが
CatBoostより構造的に弱く、ブレンドすると希釈で悪化する、という[[ensemble-oof-overfitting]]系の
負の結果が既に確定しているため）。

- **KNNブロック**: 近傍k=20人の10年定着率の平均を新しい特徴量にする（KNN局所ターゲットエンコーディング）。
  数値主要指標をPCAで10次元に圧縮した空間で距離を測る（444次元でナイーブに距離を取ると次元の呪いで
  意味をなさなくなるため）。部署Target Encodingが単一カテゴリ列ベースなのに対し、これは複数の数値特徴の
  「多変量の近さ」に基づく局所パターンを拾う。GBDTは軸並行分割なのでこの種のパターンは自力で
  再現しにくく、[[gbdt-interaction-type-matters]]の考え方の延長線上にある。
- **Coxブロック**: `data/input/employee_monthly_train_full.csv`（0-119ヶ月、既存パイプライン未使用）から
  Trainの正確な離職月を復元し、CoxPHFitterで t=120ヶ月時点の生存確率を予測、OOF特徴量として追加する。
  **入力特徴量は引き続き0-23ヶ月分のみ**（Testと条件を揃える。24ヶ月以降のデータはターゲット側
  （duration/event）の精緻化にだけ使い、特徴量には一切使わない＝リークなし）。

## リーク安全性の確認（`employee_monthly_train_full.csv`）

- Trainの退職者1,202名 = `10年定着ラベル==0`の人数と完全一致（0件の食い違い）を確認済み。
- うち129名は0-23ヶ月以内の早期退職者（[[test-set-is-survivor-filtered]]と一致）、
  残り1,073名は24-119ヶ月の間に退職しており、**これまでのパイプラインでは「いつ辞めたか」を
  一切使っていなかった**。Coxブロックはこの情報をターゲット側にのみ取り込む。

## 検証設計（1変数ずつ分離、`dept_target_enc`と同じfit_idsベースKFold OOF設計）

| 行 | 特徴量 | 何を測るか |
|---|---|---|
| Row1 | R0_memofix_plus_LM（444列、`54_`と同一） | ベースライン（再現） |
| Row2 | Row1 + KNNブロック | KNNの効果（Row2-Row1） |
| Row3 | Row1 + Coxブロック | Coxの効果（Row3-Row1） |
| Row4 | Row1 + KNN + Cox | 参考（両方同時、単独評価ではない） |

採否は[[validation-asymmetry]]の通りPublicのみで判断する。検証は足切り（`VAL_REJECT_MARGIN`超の悪化）専用。


In [1]:
!pip install -q catboost optuna lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 2.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 7.5 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from lifelines import CoxPHFitter
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [5]:
SCRIPT_NAME = "60_knn_cox_survival"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")


[2026-08-15 11:24:29] [INFO] === [60_knn_cox_survival] 実験開始 ===


INFO:60_knn_cox_survival:=== [60_knn_cox_survival] 実験開始 ===


[2026-08-15 11:24:29] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


INFO:60_knn_cox_survival:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


[2026-08-15 11:24:29] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/60_knn_cox_survival_checkpoint.csv


INFO:60_knn_cox_survival:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/60_knn_cox_survival_checkpoint.csv


[2026-08-15 11:24:29] [INFO] チェックポイントは未作成（新規実行）


INFO:60_knn_cox_survival:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")
train_monthly_full = pd.read_csv(INPUT_DIR / "employee_monthly_train_full.csv")  # 60_: 0-119ヶ月、Coxブロック専用

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")
logger.info(f"Train Monthly Full Shape: {train_monthly_full.shape} "
            f"(経過月数 {train_monthly_full['経過月数'].min()}〜{train_monthly_full['経過月数'].max()})")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")


[2026-08-15 11:24:31] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:60_knn_cox_survival:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-15 11:24:31] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:60_knn_cox_survival:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-15 11:24:31] [INFO] Train Monthly Full Shape: (257509, 29) (経過月数 0〜119)


INFO:60_knn_cox_survival:Train Monthly Full Shape: (257509, 29) (経過月数 0〜119)


[2026-08-15 11:24:31] [INFO] 定着率: 0.5647


INFO:60_knn_cox_survival:定着率: 0.5647


[2026-08-15 11:24:31] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:60_knn_cox_survival:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、Test には0名（[[test-set-is-survivor-filtered]]）。


In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。前提が崩れているので調査すること"


[2026-08-15 11:24:31] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:60_knn_cox_survival:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-15 11:24:31] [INFO] Test  早期退職者: 0名 / 2502名


INFO:60_knn_cox_survival:Test  早期退職者: 0名 / 2502名


[2026-08-15 11:24:31] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:60_knn_cox_survival:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-15 11:24:31] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:60_knn_cox_survival:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 0b. `employee_monthly_train_full.csv` からの離職月復元（60_で新規）

Coxブロックのターゲット（duration/event）を作る。0-23ヶ月の特徴量には一切触れないので
リークにはならない（詳細はセル0の説明を参照）。


In [8]:
DEP_MONTH = (
    train_monthly_full.loc[train_monthly_full["月末在籍状態"] == "退職"]
    .groupby(ID_COL)["経過月数"].min()
)

_n_events = DEP_MONTH.notna().sum() if hasattr(DEP_MONTH, "notna") else len(DEP_MONTH)
logger.info(f"離職月が判明した人数: {len(DEP_MONTH)}（うちTrainの0値ラベル人数={int((y_train==0).sum())}）")
assert len(DEP_MONTH) == int((y_train == 0).sum()), "離職月判明人数とラベル0人数が食い違う。前提が崩れている"

_dep_check = train_persona.set_index(ID_COL)[TARGET_COL].to_frame("label")
_dep_check["dep_month"] = DEP_MONTH.reindex(_dep_check.index)
_mismatch = ((_dep_check["label"] == 1) & _dep_check["dep_month"].notna()).sum() \
    + ((_dep_check["label"] == 0) & _dep_check["dep_month"].isna()).sum()
assert _mismatch == 0, f"ラベルと離職月の対応が{_mismatch}件で食い違っている"
logger.info(f"離職月の分布: min={DEP_MONTH.min():.0f}, median={DEP_MONTH.median():.0f}, max={DEP_MONTH.max():.0f}")
logger.info(f"うち24ヶ月未満（早期退職者と一致するはず）: {(DEP_MONTH < 24).sum()}名")
assert (DEP_MONTH < 24).sum() == len(EARLY_LEAVER_IDS), "24ヶ月未満の離職者数がEARLY_LEAVER_IDSと食い違う"


def build_survival_target(ids):
    """社員IDの配列から (duration, event) を作る。退職していれば実際の離職月、していなければ120で打ち切り。"""
    dep = DEP_MONTH.reindex(ids)
    duration = dep.fillna(120.0).clip(upper=120.0)
    event = dep.notna().astype(int)
    return duration.values, event.values


print("✅ Coxブロック用の生存時間ターゲット(duration/event)を復元完了")


[2026-08-15 11:24:31] [INFO] 離職月が判明した人数: 1202（うちTrainの0値ラベル人数=1202）


INFO:60_knn_cox_survival:離職月が判明した人数: 1202（うちTrainの0値ラベル人数=1202）


[2026-08-15 11:24:31] [INFO] 離職月の分布: min=11, median=53, max=119


INFO:60_knn_cox_survival:離職月の分布: min=11, median=53, max=119


[2026-08-15 11:24:31] [INFO] うち24ヶ月未満（早期退職者と一致するはず）: 129名


INFO:60_knn_cox_survival:うち24ヶ月未満（早期退職者と一致するはず）: 129名


✅ Coxブロック用の生存時間ターゲット(duration/event)を復元完了


## 1. 基本特徴量関数の定義（split非依存、`54_`と同一ロジック）

In [9]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")


✅ split非依存の基本特徴量関数定義完了


In [10]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")


[2026-08-15 11:24:31] [INFO] ------------------------------------------------------------


INFO:60_knn_cox_survival:------------------------------------------------------------


[2026-08-15 11:24:31] [INFO] split非依存の基本特徴量を生成中...


INFO:60_knn_cox_survival:split非依存の基本特徴量を生成中...


[2026-08-15 11:24:31] [INFO] ------------------------------------------------------------


INFO:60_knn_cox_survival:------------------------------------------------------------


[2026-08-15 11:30:09] [INFO] split非依存の基本特徴量生成完了


INFO:60_knn_cox_survival:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`54_`と同一・継続採用）

In [11]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")


[2026-08-15 11:30:10] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:60_knn_cox_survival:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-15 11:30:11] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:60_knn_cox_survival:入社時メモ: SVD累積寄与率=0.760


[2026-08-15 11:30:15] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:60_knn_cox_survival:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-15 11:30:16] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:60_knn_cox_survival:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-15 11:30:16] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:60_knn_cox_survival:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`54_`と同一・継続採用）

In [12]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")


[2026-08-15 11:30:16] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:60_knn_cox_survival:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-15 11:32:15] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:60_knn_cox_survival:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（`54_`と同一）

In [13]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")


[2026-08-15 11:32:16] [INFO] Persona単位の基本特徴量を生成中...


INFO:60_knn_cox_survival:Persona単位の基本特徴量を生成中...


[2026-08-15 11:32:16] [INFO] Persona単位の基本特徴量処理完了


INFO:60_knn_cox_survival:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、`49_`でパーサーを修正・`54_`と同一）

In [14]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 49_: 見出しがない書式B（276件、5.24%）のフォールバック。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")


[2026-08-15 11:32:16] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:60_knn_cox_survival:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-15 11:32:16] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:60_knn_cox_survival:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-15 11:32:16] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:60_knn_cox_survival:L_v2: Train (2761, 3), Test (2502, 3)


## 6. L2×Mリスク要因数（`54_`で確認済み、Public -0.004119・そのまま採用）

In [15]:
_ANALYTICAL_MAJOR = {"情報", "理工学"}
_ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}


def create_l2_m_interaction_features(persona_df, reloc_v2_df):
    is_analytical_major = persona_df["専攻分野"].isin(_ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(_ANALYTICAL_JOB)
    m_bad = (~is_analytical_major & is_analytical_job).astype(int)

    state = reloc_v2_df.set_index("社員ID").loc[persona_df["社員ID"], "転居x勤務地_状態_v2"].values
    l2_bad = (state == "非許容_不一致").astype(int)

    both_bad = (l2_bad & m_bad)
    risk_count = l2_bad + m_bad

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "M_不適合": m_bad,
        "L2xM_ダブル不適合": both_bad,
        "L2xM_リスク要因数": risk_count,
    })


train_l2m = create_l2_m_interaction_features(train_persona, train_reloc_v2)
test_l2m = create_l2_m_interaction_features(test_persona, test_reloc_v2)
logger.info(f"L2xMリスク特徴量: Train {train_l2m.shape}, Test {test_l2m.shape}")
print(train_l2m["L2xM_リスク要因数"].value_counts().sort_index())


[2026-08-15 11:32:16] [INFO] L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


INFO:60_knn_cox_survival:L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


L2xM_リスク要因数
0    2025
1     689
2      47
Name: count, dtype: int64


## 7. KNN局所ターゲットエンコーディング（KNNブロック、`60_`で新規追加）

数値主要指標18列をStandardScaler→PCA(10次元)で圧縮した空間で、近傍k=20人を探し、その10年定着率の
平均を新しい特徴量にする。`dept_target_encoding`と同じ fit_ids ベースのKFold OOF設計でリークを防ぐ
（fit_ids内はKFoldで自分自身の情報を除いた近傍から計算、holdout/testはfit_ids全体を近傍プールにする）。

PCA/スケーラーは fit_ids のみでfitする（[[hire-year-extrapolation]]でTestがTrainと入社時期で完全に
分断されていると分かっているため、分布の学習をfit_idsに限定するのは他ブロックと同じ慎重さ）。


In [16]:
KNN_NUMERIC_COLS = [
    "月例給与_円_mean", "残業時間_mean", "有給取得日数_mean", "欠勤日数_mean", "研修時間_mean",
    "上司との面談実施回数_mean", "情報共有件数_mean", "在宅勤務日数_mean",
    "360度評価_親和度_mean", "360度評価_信頼度_mean", "360度評価_主体度_mean",
    "360度評価_学習度_mean", "360度評価_共有貢献度_mean", "顧客満足度評価_mean", "担当プロジェクト数_mean",
    "入社時年齢", "前職経験月数", "初任給_円",
]


def create_knn_target_encoding(tf, ttf, fit_ids, seed=42, k=20, n_components=10, n_splits=5):
    """近傍k人の10年定着率の平均を新しい特徴量にする（KNN局所ターゲットエンコーディング）。"""
    y_map = train_persona.set_index(ID_COL)[TARGET_COL]

    num_cols = [c for c in KNN_NUMERIC_COLS if c in tf.columns]
    both = pd.concat([tf[[ID_COL] + num_cols], ttf[[ID_COL] + num_cols]], ignore_index=True)
    both[num_cols] = both[num_cols].fillna(both[num_cols].median())

    n_tf = len(tf)
    tf_x = both.iloc[:n_tf].reset_index(drop=True)
    ttf_x = both.iloc[n_tf:].reset_index(drop=True)

    is_fit_mask = tf_x[ID_COL].isin(fit_ids).values
    fit_idx = np.where(is_fit_mask)[0]

    scaler = StandardScaler().fit(tf_x.loc[fit_idx, num_cols])
    tf_scaled = scaler.transform(tf_x[num_cols])
    ttf_scaled = scaler.transform(ttf_x[num_cols])

    n_comp = min(n_components, len(num_cols))
    pca = PCA(n_components=n_comp, random_state=seed).fit(tf_scaled[fit_idx])
    tf_emb = pca.transform(tf_scaled)
    ttf_emb = pca.transform(ttf_scaled)

    fit_ids_arr = tf_x[ID_COL].values[fit_idx]
    fit_emb = tf_emb[fit_idx]
    fit_y = y_map.loc[fit_ids_arr].values

    train_feat = np.full(len(tf_x), np.nan)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_idx):
        k_eff = min(k, len(tr_idx))
        nn = NearestNeighbors(n_neighbors=k_eff).fit(fit_emb[tr_idx])
        _, ind = nn.kneighbors(fit_emb[val_idx])
        neighbor_y = fit_y[tr_idx][ind]
        train_feat[fit_idx[val_idx]] = neighbor_y.mean(axis=1)

    not_fit_idx = np.where(~is_fit_mask)[0]
    k_eff_full = min(k, len(fit_idx))
    nn_full = NearestNeighbors(n_neighbors=k_eff_full).fit(fit_emb)
    if len(not_fit_idx) > 0:
        _, ind_h = nn_full.kneighbors(tf_emb[not_fit_idx])
        train_feat[not_fit_idx] = fit_y[ind_h].mean(axis=1)

    _, ind_t = nn_full.kneighbors(ttf_emb)
    test_feat = fit_y[ind_t].mean(axis=1)

    train_out = pd.DataFrame({ID_COL: tf_x[ID_COL].values, "knn_local_rate": train_feat})
    test_out = pd.DataFrame({ID_COL: ttf_x[ID_COL].values, "knn_local_rate": test_feat})
    return train_out, test_out


print("✅ KNN局所ターゲットエンコーディング関数定義完了")


✅ KNN局所ターゲットエンコーディング関数定義完了


## 8. Cox生存時間特徴量（Coxブロック、`60_`で新規追加）

`CoxPHFitter`を0-23ヶ月の数値主要指標18列＋低カーディナリティのカテゴリ3列（性別・入社区分・初期等級）
で学習し、t=120ヶ月時点の生存確率をOOF特徴量にする。`duration`/`event`はセル10で復元した実際の
離職月を使う（特徴量は0-23ヶ月分のみでTestと条件を揃えており、リークはない）。

`penalizer=0.5`で正則化する。`31_`で「特徴量数がサンプル数に対し多いと弱正則化域で準分離が起きる」
という教訓が出ており、Coxの設計行列（数値18列＋ダミー変数、計30列前後）でも同じ懸念があるため。


In [17]:
COX_NUMERIC_COLS = KNN_NUMERIC_COLS  # 同じ主要指標セットを流用
COX_CAT_COLS = ["性別", "入社区分", "初期等級"]


def _build_cox_design_matrix(tf, ttf):
    num_cols = [c for c in COX_NUMERIC_COLS if c in tf.columns]
    cat_cols = [c for c in COX_CAT_COLS if c in tf.columns]

    both = pd.concat(
        [tf[[ID_COL] + num_cols + cat_cols], ttf[[ID_COL] + num_cols + cat_cols]], ignore_index=True
    )
    both[num_cols] = both[num_cols].fillna(both[num_cols].median())
    dummies = pd.get_dummies(both[cat_cols].astype(str), prefix=cat_cols, drop_first=True)
    design = pd.concat([both[[ID_COL] + num_cols].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

    n_tf = len(tf)
    return design.iloc[:n_tf].reset_index(drop=True), design.iloc[n_tf:].reset_index(drop=True)


def create_cox_survival_encoding(tf, ttf, fit_ids, seed=42, n_splits=5, penalizer=0.5):
    """CoxPHFitterで t=120 時点の生存確率をOOF特徴量として生成する。"""
    design_train, design_test = _build_cox_design_matrix(tf, ttf)
    feat_cols = [c for c in design_train.columns if c != ID_COL]

    is_fit_mask = design_train[ID_COL].isin(fit_ids).values
    fit_idx = np.where(is_fit_mask)[0]
    fit_ids_arr = design_train[ID_COL].values[fit_idx]
    duration_fit, event_fit = build_survival_target(fit_ids_arr)

    scaler = StandardScaler().fit(design_train.loc[fit_idx, feat_cols])
    X_train_scaled_df = pd.DataFrame(scaler.transform(design_train[feat_cols]), columns=feat_cols)
    X_test_scaled_df = pd.DataFrame(scaler.transform(design_test[feat_cols]), columns=feat_cols)

    train_feat = np.full(len(design_train), np.nan)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_idx):
        tr_rows = fit_idx[tr_idx]
        val_rows = fit_idx[val_idx]
        fit_df = X_train_scaled_df.iloc[tr_rows].copy().reset_index(drop=True)
        d_tr, e_tr = build_survival_target(design_train[ID_COL].values[tr_rows])
        fit_df["duration"] = d_tr
        fit_df["event"] = e_tr

        cph = CoxPHFitter(penalizer=penalizer)
        cph.fit(fit_df, duration_col="duration", event_col="event")
        sf = cph.predict_survival_function(X_train_scaled_df.iloc[val_rows], times=[120.0])
        train_feat[val_rows] = sf.loc[120.0].values

    not_fit_idx = np.where(~is_fit_mask)[0]
    fit_df_full = X_train_scaled_df.iloc[fit_idx].copy().reset_index(drop=True)
    fit_df_full["duration"] = duration_fit
    fit_df_full["event"] = event_fit
    cph_full = CoxPHFitter(penalizer=penalizer)
    cph_full.fit(fit_df_full, duration_col="duration", event_col="event")

    if len(not_fit_idx) > 0:
        sf_holdout = cph_full.predict_survival_function(X_train_scaled_df.iloc[not_fit_idx], times=[120.0])
        train_feat[not_fit_idx] = sf_holdout.loc[120.0].values

    sf_test = cph_full.predict_survival_function(X_test_scaled_df, times=[120.0])
    test_feat = sf_test.loc[120.0].values

    train_out = pd.DataFrame({ID_COL: design_train[ID_COL].values, "cox_surv120": train_feat})
    test_out = pd.DataFrame({ID_COL: design_test[ID_COL].values, "cox_surv120": test_feat})
    return train_out, test_out


print("✅ Cox生存時間特徴量関数定義完了")


✅ Cox生存時間特徴量関数定義完了


## 9. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`54_`と同一ロジックに、`extra_blocks`へ`"KNN"`/`"COX"`を追加した場合の分岐だけを足す。


In [18]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる（54_と同一 + KNN/COXブロック対応）。'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        tf = tf.merge(train_l2m, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_l2m, on=ID_COL, how="left")

    if "KNN" in extra_blocks:
        train_knn, test_knn = create_knn_target_encoding(tf, ttf, fit_ids=train_period_ids, seed=SEED)
        tf = tf.merge(train_knn, on=ID_COL, how="left")
        ttf = ttf.merge(test_knn, on=ID_COL, how="left")

    if "COX" in extra_blocks:
        train_cox, test_cox = create_cox_survival_encoding(tf, ttf, fit_ids=train_period_ids, seed=SEED)
        tf = tf.merge(train_cox, on=ID_COL, how="left")
        ttf = ttf.merge(test_cox, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（KNN/COXブロック対応版）")


✅ 部署Target Encoding・prepare_split関数定義完了（KNN/COXブロック対応版）


## 10. 特徴量の組み立て（extra_blocks={"L2","KNN","COX"}を常時マージし、CONFIGSで列選択を切替）

In [19]:
def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


BLOCK = {"L2", "KNN", "COX"}

logger.info("=" * 60)
logger.info("[検証用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[提出用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"main_train={len(ag_train_80b)}, main_valid(生存者)={len(ag_val_surv)}")
logger.info(f"全件={len(ag_full)}")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80b))}")
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"


[2026-08-15 11:32:16] [INFO] ============================================================


INFO:60_knn_cox_survival:============================================================


[2026-08-15 11:32:16] [INFO] [検証用] split_80_20 / 検証=生存者のみ


INFO:60_knn_cox_survival:[検証用] split_80_20 / 検証=生存者のみ


[2026-08-15 11:32:17] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


...

INFO:60_knn_cox_survival:main_train=2208, main_valid(生存者)=535


[2026-08-15 11:32:18] [INFO] 全件=2761


INFO:60_knn_cox_survival:全件=2761


[2026-08-15 11:32:18] [INFO] 特徴量数: 446


INFO:60_knn_cox_survival:特徴量数: 446


## 11. 特徴量グループの棚卸し（KNN/COXグループを追加）

In [20]:
ALL_FEATS = set(_feature_cols(ag_train_80b))

DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "LM":        _cols_of(train_l2m),
    "KNN":       ["knn_local_rate"],
    "COX":       ["cox_surv120"],
    "derived":   DERIVED_COLS,
}

FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 446 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  LM            3 列   例: ['M_不適合', 'L2xM_ダブル不適合']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
  KNN           1 列   例: ['knn_local_rate']
  COX           1 列  

## 12. 構成の事前登録

`54_`のA_PARAMS/反復数をそのまま流用する（ハイパラ探索はしない＝「特徴量だけの差」にするため）。


In [21]:
A_PARAMS = {
    "depth": 4,
    "learning_rate": 0.03518359458951149,
    "l2_leaf_reg": 2.217690447016724,
    "border_count": 218,
    "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}

ITER_HOLDOUT = 560   # 38_・54_と同一
ITER_FULL    = 560

SEEDS_SUB = [42, 2024, 7, 1234, 99]
SEEDS_VAL = [42, 2024, 7, 1234, 99, 555, 31337, 2718]

CONFIGS = {
    "R0_memofix_plus_LM":         {"groups": ALL_GROUPS - {"KNN", "COX"}},  # 54_のベスト構成の再現（444列）
    "R0_plus_LM_plus_KNN":        {"groups": ALL_GROUPS - {"COX"}},         # + KNN局所ターゲットエンコーディング
    "R0_plus_LM_plus_COX":        {"groups": ALL_GROUPS - {"KNN"}},        # + Cox生存確率
    "R0_plus_LM_plus_KNN_COX":    {"groups": ALL_GROUPS},                   # 参考: 両方同時
}

VAL_REJECT_MARGIN = 0.02


def cols_for(spec, df):
    keep = set()
    for g in spec["groups"]:
        keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


print(f"{'config':<28s} {'列数':>5s}")
print("-" * 44)
for name, spec in CONFIGS.items():
    print(f"{name:<28s} {len(cols_for(spec, ag_train_80b)):>5d}")


config                          列数
--------------------------------------------
R0_memofix_plus_LM             444
R0_plus_LM_plus_KNN            445
R0_plus_LM_plus_COX            445
R0_plus_LM_plus_KNN_COX        446


## 13. モデル関数（標準パイプライン、`54_`と同一。反復数固定・early stoppingなし）

In [22]:
def _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed):
    model = cb.CatBoostClassifier(
        **params, iterations=int(n_iter), random_seed=seed,
        verbose=False, cat_features=obj_cols, task_type="CPU",
    )
    model.fit(X_tr, y_tr)
    return model


def fit_holdout_fixed(ag_train, ag_val, feature_cols, params, n_iter, seeds):
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = ag_train[feature_cols].fillna(-999), ag_train[TARGET_COL]
    X_va, y_va = ag_val[feature_cols].fillna(-999), ag_val[TARGET_COL]

    val_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed)
        val_preds.append(model.predict_proba(X_va)[:, 1])
    val_preds = np.array(val_preds)

    singles = [log_loss(y_va, vp) for vp in val_preds]
    return {
        "val_seedavg": float(log_loss(y_va, val_preds.mean(axis=0))),
        "val_single_mean": float(np.mean(singles)),
        "val_single_sd": float(np.std(singles)),
        "val_preds": val_preds,
        "y_val": y_va.values,
    }


def fit_full_fixed(ag_full, test_feats, feature_cols, params, n_iter, seeds):
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = ag_full[feature_cols].fillna(-999), ag_full[TARGET_COL]
    X_test = test_feats[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"    seed={seed}: 全件学習完了")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)

print("✅ モデル関数定義完了（fit_holdout_fixed / fit_full_fixed）")


✅ モデル関数定義完了（fit_holdout_fixed / fit_full_fixed）


## 14. チェックポイント

In [23]:
RESULT_SCHEMA = ["config", "kind", "n_features", "val_seedavg", "val_single_mean", "val_single_sd",
                  "n_iterations", "n_train", "pred_mean", "submission_path"]


def make_row(**kwargs):
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)


def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)


def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label] if len(checkpoint) else checkpoint
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")


✅ チェックポイント関数定義完了


## 15. 4構成の実行

In [24]:
def make_standard_runner(config_label, spec):
    def _run():
        feats = cols_for(spec, ag_train_80b)
        feats_full = cols_for(spec, ag_full)
        assert feats == feats_full, "検証と全件学習で特徴量列が食い違っている"

        logger.info("=" * 60)
        logger.info(f"[{config_label}] {len(feats)}列")

        hold = fit_holdout_fixed(ag_train_80b, ag_val_surv, feats, A_PARAMS, ITER_HOLDOUT, SEEDS_VAL)
        logger.info(f"  検証(生存者{len(ag_val_surv)}名): シード平均 {hold['val_seedavg']:.6f} "
                    f"/ 単一シード {hold['val_single_mean']:.6f} ± {hold['val_single_sd']:.6f}")
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy", hold["val_preds"])

        test_preds = fit_full_fixed(ag_full, test_features_full, feats, A_PARAMS, ITER_FULL, SEEDS_SUB)
        preds = test_preds.mean(axis=0)
        path = save_submission(test_features_full.index, preds, config_label)

        return make_row(
            config=config_label, kind="standard", n_features=len(feats),
            val_seedavg=hold["val_seedavg"], val_single_mean=hold["val_single_mean"],
            val_single_sd=hold["val_single_sd"],
            n_iterations=ITER_FULL, n_train=len(ag_full), pred_mean=float(preds.mean()),
            submission_path=path,
        )
    return _run


standard_results = {}
for _name, _spec in CONFIGS.items():
    standard_results[_name] = run_or_resume(_name, make_standard_runner(_name, _spec))

print()
print(f"{'config':<28s} {'列数':>5s} {'val(8シード平均)':>16s} {'単一sd':>9s}")
print("-" * 62)
for _name, _r in standard_results.items():
    print(f"{_name:<28s} {int(_r['n_features']):>5d} {float(_r['val_seedavg']):>16.6f} {float(_r['val_single_sd']):>9.6f}")


[2026-08-15 11:32:19] [INFO] ============================================================


INFO:60_knn_cox_survival:============================================================


[2026-08-15 11:32:19] [INFO] [R0_memofix_plus_LM] 444列


INFO:60_knn_cox_survival:[R0_memofix_plus_LM] 444列


[2026-08-15 11:32:59] [INFO]   検証(生存者535名): シード平均 0.505477 / 単一シード 0.508987 ± 0.005181


...

[2026-08-15 11:36:39] [INFO]     seed=99: 全件学習完了


INFO:60_knn_cox_survival:    seed=99: 全件学習完了


[2026-08-15 11:36:39] [INFO]   提出ファイル: 20260815_60_knn_cox_survival_R0_plus_LM_plus_KNN_COX.csv（予測平均=0.5918）


INFO:60_knn_cox_survival:  提出ファイル: 20260815_60_knn_cox_survival_R0_plus_LM_plus_KNN_COX.csv（予測平均=0.5918）



config                          列数      val(8シード平均)      単一sd
--------------------------------------------------------------
R0_memofix_plus_LM             444         0.505477  0.005181
R0_plus_LM_plus_KNN            445         0.506556  0.005632
R0_plus_LM_plus_COX            445         0.506460  0.002127
R0_plus_LM_plus_KNN_COX        446         0.506306  0.003487


## 16. 結果まとめ

In [25]:
baseline_val = float(standard_results["R0_memofix_plus_LM"]["val_seedavg"])

rows = []
for name, r in standard_results.items():
    val = float(r["val_seedavg"])
    rows.append({"config": name, "列数": int(r["n_features"]), "val": val,
                 "val差(対baseline)": val - baseline_val,
                 "ファイル": Path(r["submission_path"]).name})
summary = pd.DataFrame(rows).set_index("config")

summary["提出"] = "提出する"
for name in ["R0_plus_LM_plus_KNN", "R0_plus_LM_plus_COX"]:
    if summary.loc[name, "val差(対baseline)"] > VAL_REJECT_MARGIN:
        summary.loc[name, "提出"] = "見送り（足切り）"
summary.loc["R0_plus_LM_plus_KNN_COX", "提出"] = "参考（単独評価ではない・任意）"

pd.set_option("display.width", 200)
print()
print(summary.to_string())
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv")
logger.info(f"サマリを保存: {TODAY}_{SCRIPT_NAME}_summary.csv")



                          列数       val  val差(対baseline)                                                      ファイル               提出
config                                                                                                                            
R0_memofix_plus_LM       444  0.505477         0.000000       20260815_60_knn_cox_survival_R0_memofix_plus_LM.csv             提出する
R0_plus_LM_plus_KNN      445  0.506556         0.001078      20260815_60_knn_cox_survival_R0_plus_LM_plus_KNN.csv             提出する
R0_plus_LM_plus_COX      445  0.506460         0.000983      20260815_60_knn_cox_survival_R0_plus_LM_plus_COX.csv             提出する
R0_plus_LM_plus_KNN_COX  446  0.506306         0.000829  20260815_60_knn_cox_survival_R0_plus_LM_plus_KNN_COX.csv  参考（単独評価ではない・任意）
[2026-08-15 11:36:39] [INFO] サマリを保存: 20260815_60_knn_cox_survival_summary.csv


INFO:60_knn_cox_survival:サマリを保存: 20260815_60_knn_cox_survival_summary.csv


## 17. 提出方針

### 判定（事前登録・[[validation-asymmetry]]と同様の考え方）

- **採否は Public のみ。** 検証は足切り（悪化検出）専用。
- **足切り**: `VAL_REJECT_MARGIN`（0.02）を超えて悪化したら提出しない。
- Row2（KNN）とRow3（Cox）は、Row1という同一のベースラインに対する**単一の事前登録済み介入**なので、
  両方提出してよい（複数構成から検証スコアで選ぶ探索ではない）。Row4（両方同時）は参考情報として記録するが、
  単独の効果測定ではないため採否判断には使わない。

### 期待値について

- どちらも[[kitchen-sink-combination-search]]で「既存ブロックの組み合わせでは新規性なし」と確定した
  結論の対象外の、構造的に新しいタイプの特徴量（多変量の近さ・生存時間モデルの出力）なので試す価値はある。
- ただし`31_`/`32_`の教訓（線形モデル単体はCatBoostよりかなり弱く、確率ブレンドは希釈で悪化する）から、
  Coxブロックは「Cox単体で予測する」のではなく「Coxの出力を1特徴量としてCatBoostに渡す」設計にしている。
  この設計変更がTrack3の失敗パターンを回避できるかどうかも、今回の検証で分かることの一つ。
